# שיעור 08 - תבנית עיצוב מרובי סוכנים


## הגדרה


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## מדוע מערכות רב-סוכנים?

משימות מהעולם האמיתי כמו תכנון טיול כוללות סוגים רבים ושונים של מומחיות — לוגיסטיקה, ידע מקומי, תקצוב ועוד. סוכן יחיד שמנסה לטפל בכל quickly הופך לבלתי ניתן לניהול.

מערכות רב-סוכנים פותרות זאת באמצעות **התמחות**: כל סוכן מתמקד בתחום מומחיות אחד, ומפיק תוצאות באיכות גבוהה יותר מאשר גנרליסט. הן גם משפרות את ה**היקף** — ניתן להוסיף סוכנים חדשים (למשל, מומחה טיסות, מבקר מסעדות) מבלי לשכתב את זרימת העבודה הקיימת. הסוכנים משתלבים יחד דרך צינור מובנה, מעבירים הקשר מאחד לשני.


## יצירת סוכנים מומחים


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## בניית זרימת עבודה רציפה

`WorkflowBuilder` מאפשר לך לקשר סוכנים לגרף מכוון. כאן אנו יוצרים קו צינור פשוט עם שני שלבים: ה-**TravelPlanner** מנסח את מסלול הנסיעה, ואז ה-**TravelConcierge** בודק ומשפר אותו.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## הוספת סוכנים נוספים לזרימת העבודה

אחת מהיתרונות הגדולים ביותר של תבנית הסוכנים המרובים היא כמה קל להרחיב אותה. למטה אנו מוסיפים סוכן **BudgetReviewer** שבודק את התכנית מול התקציב של המטייל, מסמן פריטים שעשויים לדחוף את העלויות מעבר למגבלה, ומציע אפשרויות לחיסכון בכסף. זרימת העבודה עכשיו מריצה שלושה סוכנים ברצף:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## סיכום

בתרגיל זה למדת כיצד:

1. **ליצור סוכנים מתמחים** — כל אחד עם תפקיד ממוקד (תכנון, קונסיירז', סקירת תקציב).
2. **לחבר סוכנים לתהליך עבודה סדרתי** באמצעות `WorkflowBuilder` ו- `add_edge`.
3. **לשדר פלט** מתוך צינור רב-סוכני, ולעקוב איזה סוכן מדבר.
4. **להרחיב תהליך עבודה** על ידי הוספת סוכנים חדשים לשרשרת מבלי לשנות את הקיימים.

תבנית עיצוב רב-סוכנים שומרת על כל סוכן פשוט, תוך ייצור תוצאות עשירות ומבוקרות לעומק יותר מאשר כל סוכן יחיד יכול להשיג לבדו.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש להיות מודעים לכך שתרגומים אוטומטיים עשויים להכיל טעויות או אי-דיוקים. המסמך המקורי בשפתו המקורית צריך להיחשב כמקור המוסמך. למידע קריטי מומלץ להשתמש בתרגום מקצועי אנושי. איננו נושאים באחריות לכל אי-הבנות או פרשנויות שגויות הנובעות מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
